In [2]:
import duckdb
import os
import time

VOCAB_DIR = "../data/omop_vocab"
DB_PATH = "../data/omop_clinical.duckdb"

def load_concept_table():
    concept_csv = os.path.join(VOCAB_DIR, "CONCEPT.csv")
    
    if not os.path.exists(concept_csv):
        print(f"❌ Erro: O ficheiro {concept_csv} não existe.")
        return

    print("🔌 A ligar ao DuckDB...")
    
    try:
        # Abrimos a ligação e garantimos que ela se fecha quer o código falhe quer corra bem
        con = duckdb.connect(DB_PATH)
        
        print("⏳ A carregar a tabela CONCEPT... (O DuckDB é rápido, mas pode demorar uns segundos)")
        start_time = time.time()
        
        # Elimina a tabela se ela ficou a meio num carregamento anterior
        con.execute("DROP TABLE IF EXISTS concept")
        
        # Carrega os dados diretamente do CSV para a base de dados em disco
        con.execute(f"""
            CREATE TABLE concept AS 
            SELECT * FROM read_csv_auto('{concept_csv}', header=True, delim='\t', nullstr='', sample_size=100000)
        """)
        
        elapsed_time = time.time() - start_time
        
        # Conta as linhas sem passar para o Pandas
        count = con.execute("SELECT COUNT(*) FROM concept").fetchone()[0]
        print(f"✅ Sucesso! Foram carregados {count:,} conceitos em {elapsed_time:.2f} segundos.")
        
        # Mostra 5 resultados nativamente (sem usar fetchdf)
        print("\n🔎 Amostra de diagnósticos SNOMED carregados:")
        sample = con.execute("""
            SELECT concept_id, concept_name, concept_class_id 
            FROM concept 
            WHERE vocabulary_id = 'SNOMED' AND concept_class_id = 'Clinical Finding'
            LIMIT 5
        """).fetchall()
        
        for row in sample:
            print(f" - ID: {row[0]} | Name: {row[1]} | Class: {row[2]}")
            
    except Exception as e:
        print(f"❌ Erro crítico durante o carregamento: {e}")
        
    finally:
        # Isto é a regra de ouro: fechar SEMPRE a ligação no fim
        con.close()
        print("\n🔒 Ligação ao DuckDB fechada com segurança.")

# Executa o carregamento
load_concept_table()

🔌 A ligar ao DuckDB...
⏳ A carregar a tabela CONCEPT... (O DuckDB é rápido, mas pode demorar uns segundos)
✅ Sucesso! Foram carregados 6,456,570 conceitos em 10.58 segundos.

🔎 Amostra de diagnósticos SNOMED carregados:
 - ID: 42538812 | Name: Somatic hallucination | Class: Clinical Finding
 - ID: 40629514 | Name: Stillbirth | Class: Clinical Finding
 - ID: 40277853 | Name: Small visual image | Class: Clinical Finding
 - ID: 40304440 | Name: Pain: [site of GIT] or [abdominal site symptom] or [flank] or [subcostal] or [iliac fossa] | Class: Clinical Finding
 - ID: 40304969 | Name: Micturition stream normal | Class: Clinical Finding

🔒 Ligação ao DuckDB fechada com segurança.
